# COMP3027 Week 2: Greedy Algorithm
This notebook covers:
- The greedy paradigm and how to reason about greedy correctness.
- Fractional knapsack: multiple greedy rules, counterexamples, and the optimal density-based greedy algorithm.

In [45]:
from dataclasses import dataclass
from typing import List, Tuple, Callable, Dict
import random
import heapq
import itertools

## 1. Greedy Algorithms — Introduction

### 1.1 Greedy paradigm
A greedy algorithm constructs a solution incrementally. At each step it makes a locally optimal choice
according to some rule, hoping this leads to a globally optimal solution.

Generic structure:
1. Maintain a partial solution `S`.
2. Repeatedly choose the "best" next element `e` (greedy choice).
3. Update `S` and the remaining subproblem.
4. Stop when complete.

**Key difficulty:** the greedy rule is not automatically correct; we must prove it.

---

### 1.2 Common correctness proof pattern (exchange argument)
To prove a greedy algorithm is optimal, a common plan is:

1. Let `g1` be the first greedy choice.
2. Take an arbitrary optimal solution `OPT`.
3. Show we can transform `OPT` into another optimal solution `OPT'` that includes `g1`
   without worsening the objective (this is the "exchange" step).
4. After fixing `g1`, the remaining part becomes a smaller subproblem.
5. Use induction to conclude the whole greedy solution is optimal.

We'll see this directly in fractional knapsack.

---
## 2. Fractional Knapsack

### 2.1 Problem formulation
We have items `i = 1..n` with:
- value (benefit) `b_i > 0`
- weight `w_i > 0`
Knapsack capacity `W > 0`.

Fractional means we can take a fraction `x_i ∈ [0,1]` of each item.

Optimization problem:

$$
\max \sum_{i=1}^n b_i x_i
\quad \text{s.t.} \quad
\sum_{i=1}^n w_i x_i \le W,\;\; 0 \le x_i \le 1.
$$

Define **density**:
$$
\rho_i = \frac{b_i}{w_i}.
$$

Claim: the optimal greedy rule is to take items in **descending density**.

In [46]:
@dataclass(frozen=True)
class Item:
    value: float
    weight: float

    @property
    def density(self) -> float:
        if self.weight <= 0:
            raise ValueError('Weight must be positive')
        return self.value / self.weight

def total_weight(items: List[Item], plan: List[Tuple[int, float]]) -> float:
    return sum(items[i].weight * frac for i, frac in plan)

def total_value(items: List[Item], plan: List[Tuple[int, float]]) -> float:
    return sum(items[i].value * frac for i, frac in plan)

### 2.2 Greedy rule candidates
Common "intuitive" greedy ideas:

1. Take the item with **highest value** first.
2. Take the item with **lowest weight** first.
3. Take the item with **highest density** $\rho_i = b_i / w_i$ first.

For fractional knapsack, only **(3)** is always optimal.

We'll implement all three and find counterexamples for (1) and (2).

In [47]:
def greedy_fractional_knapsack(
    items: List[Item],
    capacity: float,
    key: Callable[[Item], float],
    reverse: bool = True,
) -> Tuple[float, List[Tuple[int, float]]]:
    """
    Generic greedy fractional knapsack algorithm.
    Sort by key(item), then take as much as possible in that order.
    Returns (max_value, plan) where plan = [(idx, fraction)].
    """
    if capacity < 0:
        raise ValueError('Capacity must be non-negative')

    indexed = [(idx, item) for idx, item in enumerate(items)]
    indexed.sort(key=lambda x: key(x[1]), reverse=reverse)

    remaining_capacity = capacity
    plan: List[Tuple[int, float]] = []
    value = 0.0

    for idx, item in indexed:
        if remaining_capacity <= 0:
            break
        if item.weight <= remaining_capacity:
            plan.append((idx, 1.0))
            remaining_capacity -= item.weight
            value += item.value
        else:
            frac = remaining_capacity / item.weight
            plan.append((idx, frac))
            value += item.value * frac
            remaining_capacity = 0.0

    return value, plan

def greedy_by_value(items: List[Item], W: float):
    return greedy_fractional_knapsack(
        items, W, key=lambda item: item.value, reverse=True
    )

def greedy_by_weight(items: List[Item], W: float):
    return greedy_fractional_knapsack(
        items, W, key=lambda item: item.weight, reverse=True
    )

def greedy_by_density(items: List[Item], W: float):
    return greedy_fractional_knapsack(
        items, W, key=lambda item: item.density, reverse=True
    )

### 2.3 Optimal greedy algorithm (density-based)

Algorithm:
1. Compute density $\rho_i = b_i / w_i$.
2. Sort items by $\rho_i$ decreasing.
3. Take items in that order, taking full items until the last one (possibly fractional).

Time complexity:
- Sorting: $O(n\log n)$
- One pass fill: $O(n)$

Total: $O(n\log n)$.

In [48]:
def fractional_knapsack_optimal(items: List[Item], capacity: float):
    return greedy_by_density(items, capacity)

# quick test
items = [
    Item(value=60, weight=10),
    Item(value=100, weight=20),
    Item(value=120, weight=30)
]
W = 50
best_value, best_plan = fractional_knapsack_optimal(items, W)
best_value, best_plan

(240.0, [(0, 1.0), (1, 1.0), (2, 0.6666666666666666)])

### 2.4 Why density greedy is optimal (exchange argument)

Assume items are sorted so that:
$$
\rho_1 \ge \rho_2 \ge \dots \ge \rho_n,\quad \rho_i = \frac{b_i}{w_i}.
$$

Consider any feasible solution $\mathbf{x} = \begin{pmatrix}x_1 & \cdots & x_n\end{pmatrix}^\top\in[0,1]^n$ and total weight $\sum_{i=1}^nw_ix_i\le W$.

Suppose $\mathbf{x}$ is optimal but **not** density-ordered.
Then there exist indices $i < k$ such that:
- $x_i < 1$ (we didn't fully take a higher-density item),
- $x_k > 0$ (we took some of a lower-density item),
- and $\rho_i > \rho_k$.

Let $\delta > 0$ be a small amount of weight that we shift from item \(k\) to item \(i\):
- decrease $x_k$ by $\delta / w_k$,
- increase $x_i$ by $\delta / w_i$.

**Feasibility:** total weight stays the same:
$$
w_i\left(\frac{\delta}{w_i}\right) - w_k\left(\frac{\delta}{w_k}\right)=\delta-\delta=0.
$$

**Objective improvement:**
$$
\Delta
= b_i\left(\frac{\delta}{w_i}\right) - b_k\left(\frac{\delta}{w_k}\right)
= \delta\left(\frac{b_i}{w_i} - \frac{b_k}{w_k}\right)
= \delta(\rho_i - \rho_k) > 0.
$$

So we strictly improve the objective — contradiction.
Therefore, any optimal solution must take higher-density items before lower-density ones, which is exactly what greedy does.

In [49]:
def random_items(n: int, v_range=(1, 200), w_range=(1, 50)) -> List[Item]:
    return [Item(value=random.randint(*v_range), weight=random.randint(*w_range)) for _ in range(n)]

def compare_heuristics(trials=200, n=8, W=80):
    worse_val = 0
    worse_wt = 0
    for _ in range(trials):
        items = random_items(n)
        v_den, _ = greedy_by_density(items, W)
        v_val, _ = greedy_by_value(items, W)
        v_wt, _ = greedy_by_weight(items, W)
        if v_val > v_den + 1e-9:
            worse_val += 1
        if v_wt > v_den + 1e-9:
            worse_wt += 1
    return worse_val, worse_wt

worse_val, worse_wt = compare_heuristics()
print(f"worse_val: {worse_val}, worse_wt: {worse_wt}")

worse_val: 0, worse_wt: 0


It shows that density greedy is always optimal compared to value/weight greedy.

---
## 3. Interval Scheduling (Max number of non-overlapping intervals)

### 3.1 Problem
We have `n` intervals/jobs. Job `j` has a start time `s_j` and finish time `f_j` with `s_j < f_j`.

Two jobs are compatible if they do not overlap.

Goal: select a maximum-cardinality subset of mutually compatible jobs.

---

### 3.2 Greedy rule candidates
Common intuitive greedy ideas (some are wrong!):

1. Pick the job with **earliest start time**.
2. Pick the job with **shortest duration** (`f - s`).
3. Pick the job with **earliest finish time**.

Claim: Only **earliest finish time** is always optimal.

In [50]:
@dataclass(frozen=True)
class Interval:
    start: float
    finish: float
    name: str = ""

    @property
    def duration(self) -> float:
        return self.finish - self.start

def is_compatible(a: Interval, b: Interval) -> bool:
    # assume we schedule in time order; compatibility if non-overlapping
    return a.finish <= b.start or b.finish <= a.start

In [51]:
def greedy_interval_scheduling(
    intervals: List[Interval],
    key: Callable[[Interval], float],
    reverse: bool = False,
) -> List[int]:
    """
    Returns indices of selected intervals (by original indices) using a greedy rule:
    - sort intervals by key
    - iterate, picking an interval if it starts after (or at) last chosen finish
    This template works for earliest-finish-time greedy (optimal).
    """
    indexed = list(enumerate(intervals))
    indexed.sort(key=lambda x: key(x[1]), reverse=reverse)

    chosen: List[int] = []
    last_finish = None

    for idx, interval in indexed:
        if last_finish is None or interval.start >= last_finish:
            chosen.append(idx)
            last_finish = interval.finish

    return chosen

def greedy_earliest_start(intervals: List[Interval]) -> List[int]:
    return greedy_interval_scheduling(
        intervals,
        key=lambda i: i.start,
        reverse=False
    )

def greedy_earliest_duration(intervals: List[Interval]) -> List[int]:
    return greedy_interval_scheduling(
        intervals,
        key=lambda i: i.duration,
        reverse=False
    )

def greedy_earliest_finish(intervals: List[Interval]) -> List[int]:
    return greedy_interval_scheduling(
        intervals,
        key=lambda i: i.finish,
        reverse=False
    )

def describe_schedule(intervals: List[Interval], chosen: List[int]) -> None:
    chosen_sorted = sorted(chosen, key=lambda i: intervals[i].finish)
    print("Chosen:", [intervals[i].name or i for i in chosen_sorted])
    for i in chosen_sorted:
        interval = intervals[i]
        print(f"{interval.name or i}: [{interval.start}, {interval.finish}]  duration={interval.duration}")
    print("count =", len(chosen_sorted))

In [52]:
# Earliest-start-time fails:
# If you pick the long interval that starts earliest, you may block many short ones.
intervals = [
    Interval(start=0, finish=10, name="A_long"),
    Interval(start=1, finish=2, name="B"),
    Interval(start=2, finish=3, name="C"),
    Interval(start=3, finish=4, name="D"),
    Interval(start=4, finish=5, name="E"),
    Interval(start=5, finish=6, name="F"),
]

chosen_start = greedy_earliest_start(intervals)
chosen_finish = greedy_earliest_finish(intervals)

print("Earliest start greedy:")
describe_schedule(intervals, chosen_start)

print("\nEarliest finish greedy:")
describe_schedule(intervals, chosen_finish)

Earliest start greedy:
Chosen: ['A_long']
A_long: [0, 10]  duration=10
count = 1

Earliest finish greedy:
Chosen: ['B', 'C', 'D', 'E', 'F']
B: [1, 2]  duration=1
C: [2, 3]  duration=1
D: [3, 4]  duration=1
E: [4, 5]  duration=1
F: [5, 6]  duration=1
count = 5


Shortest-duration picks `tiny_cross` first, then blocks `A` and `B`, reducing total count compared to earliest-finish.

---
### 3.3 Optimal greedy algorithm: earliest finish time

Algorithm:
1. Sort intervals by increasing finish time.
2. Scan in that order, and take an interval if it is compatible with the last chosen one.

Time complexity:
- Sorting: $O(n\log n)$
- Scan: $O(n)$

Total: $O(n\log n)$.

---
### 3.4 Correctness proof (exchange argument + induction)

Let the greedy algorithm pick the interval $F$ with **earliest finish time**.

**Greedy-choice property.**
There exists an optimal solution that includes $F$.

*Proof idea (exchange).*
Let $O$ be an optimal solution, and let $J$ be the first interval in $O$ (the one with the smallest finish time among intervals in $O$).
Since $F$ has the earliest finish among all intervals, we have:
$$
f(F) \le f(J).
$$
Replace $J$ by $F$. This replacement keeps feasibility:
- $F$ ends no later than $J$, so any interval that starts after $f(J)$ also starts after $f(F)$.

Thus we obtain a schedule with the same number of intervals, still optimal, that includes $F$.

**Optimal substructure.**
After choosing $F$, the remaining problem is to schedule among intervals that start after $f(F)$. Call that set $C$.
An optimal solution for the whole problem is:
$$
\{F\} \cup \text{OPT}(C).
$$

**Induction.**
Greedy solves the smaller instance $C$ optimally by the same argument. Hence the whole greedy schedule is optimal.

---
### 3.5 Quick randomised "sanity check" vs brute force (small n)
For a small $n$, we can brute force the true optimum to validate the greedy.

In [53]:
def is_schedule_valid(intervals: List[Interval], subset: List[int]) -> bool:
    chosen = [intervals[i] for i in subset]
    chosen.sort(key=lambda i: i.start)
    for i in range(1, len(chosen)):
        if chosen[i].start < chosen[i-1].finish:
            return False
    return True

def brute_force_optimum(intervals: List[Interval]) -> List[int]:
    n = len(intervals)
    best: List[int] = []
    for r in range(n+1):
        for subset in itertools.combinations(range(n), r):
            subset = list(subset)
            if is_schedule_valid(intervals, subset) and len(subset) > len(best):
                best = subset
    return best

def random_intervals(n: int, tmax=20) -> List[Interval]:
    ints = []
    for i in range(n):
        a = random.randint(0, tmax-1)
        b = random.randint(a+1, tmax)
        ints.append(Interval(a, b, f"I{i}"))
    return ints

In [54]:
trials = 200
n = 10
bad = 0
for _ in range(trials):
    ints = random_intervals(n)
    g = greedy_earliest_finish(ints)
    opt = brute_force_optimum(ints)
    if len(g) != len(opt):
        bad += 1

print(f"{bad} bad cases")

0 bad cases


## 4. Scheduling to Minimize Maximum Lateness (EDF)

### 4.1 Problem
Single machine, non-preemptive. We must schedule all jobs (no dropping).

Each job $i$ has:
- processing time $t_i > 0$
- deadline $d_i$

A schedule is an ordering of jobs. Let $s_i$ be start time and $f_i = s_i + t_i$ be finish time.

Define lateness:
$$
\ell_i = \max\{0, f_i - d_i\}.
$$

Objective: minimize maximum lateness:
$$
\min L \quad \text{where} \quad L = \max_i \ell_i.
$$

We will prove the optimal greedy rule is **EDF (Earliest Deadline First)**:
sort by nondecreasing deadlines $d_i$.

In [55]:
@dataclass(frozen=True)
class Job:
    process_time: int
    ddl: int
    name: str = ""

def evaluate_schedule(
    jobs: List[Job],
    order: List[int]
) -> Tuple[int, List[Tuple[str, int, int, int]]]:
    """
    Returns:
        L: maximum lateness
        details: list of (job_name, start, finish, lateness)
    """
    time = 0
    L = 0
    details = []
    for idx in order:
        job = jobs[idx]
        start = time
        finish = time + job.process_time
        lateness = max(0, finish - job.ddl)
        L = max(L, lateness)
        details.append((job.name or str(idx),
                        start, finish, lateness))
        time = finish
    return L, details

def print_schedule(jobs: List[Job], order: List[int]) -> None:
    L, details = evaluate_schedule(jobs, order)
    print("OrderL", [jobs[i].name or i for i in order])
    for name, s, f, l in details:
        print(f"  {name:8s} start={s:>3} finish={f:>3} \
        deadline={next(j.ddl for j in jobs if (j.name or str(jobs.index(j)))==name)}")
    print("Max latrness L =", L)

def print_schedule_table(jobs: List[Job], order: List[int]) -> None:
    L, details = evaluate_schedule(jobs, order)
    print("Order:", [jobs[i].name or i for i in order])
    print(f"{'job':<8}{'t':>4}{'d':>5}{'start':>7}{'finish':>8}{'late':>7}")
    time = 0
    for idx in order:
        j = jobs[idx]
        start = time
        finish = time + j.process_time
        late = max(0, finish - j.ddl)
        print(f"{(j.name or str(idx)):<8}{j.process_time:>4}{j.ddl:>5}{start:>7}{finish:>8}{late:>7}")
        time = finish
    print("Max lateness L =", L)

def edf_order(jobs: List[Job]) -> List[int]:
    # earliest deadline first
    return sorted(range(len(jobs)), key=lambda i: jobs[i].ddl)

def shortest_processing_time_order(jobs: List[Job]) -> List[int]:
    return sorted(range(len(jobs)), key=lambda i: jobs[i].process_time)

def latest_deadline_first_order(jobs: List[Job]) -> List[int]:
    return sorted(range(len(jobs)), key=lambda i: jobs[i].ddl, reverse=True)

### 4.2 Greedy algorithm: EDF (Earliest Deadline First)

Algorithm:
1. Sort jobs so that $d_1 \le d_2 \le \dots \le d_n$.
2. Schedule them back-to-back in that order (no idle time).

Time complexity: $O(n\log n)$ due to sorting.

---
### 4.3 Structural facts used in the proof

**Fact 1 (No idle time in an optimal schedule).**
There exists an optimal schedule with no idle time.
Reason: if the machine is idle while jobs remain, shifting later jobs earlier cannot increase any completion time, hence cannot increase max lateness.

**Fact 2 (Inversions).**
Consider any schedule. A pair of jobs $i, k$ forms an *inversion* if:
- $d_i < d_k$ (job $i$ has earlier deadline),
- but $k$ is scheduled before $i$.

EDF schedules have **no inversions**.

---
### 4.4 Correctness proof (inversion + adjacent swap argument)

We prove EDF minimizes $L = \max\{\max\{0, f_i - d_i\}:i=1,\ldots,n\}$.

Take an optimal schedule $S^*$ with **no idle time** and with the **fewest inversions** among all optimal schedules.
If $S^*$ has no inversions, then jobs are already sorted by non-decreasing deadlines → it is an EDF schedule.

So assume for contradiction that $S^*$ has at least one inversion.
Then there exists an **adjacent inversion**: two consecutive jobs in the schedule, say job $k$ followed immediately by job $i$,
such that $d_i < d_k$.

Let $T$ be the time just before these two jobs start.

- In the original schedule:
$$
f_k = T + t_k,\qquad f_i = T + t_k + t_i.
$$

- After swapping their order (put $i$ before $k$):
$$
f'_i = T + t_i,\qquad f'_k = T + t_i + t_k.
$$

All other jobs’ completion times are unchanged, so only lateness of $i$ and $k$ matters.

Now compare maximum lateness before vs after the swap:

1) Job $i$ moves earlier, so $f'_i \le f_i$, hence $\ell'_i \le \ell_i$.

2) Job $k$ moves later, but $k$ has the *later deadline* ($d_k > d_i$).
The swap can be shown not to increase the maximum lateness because any "damage" to $k$ is cushioned by its later deadline,
while the benefit to $i$ prevents a potentially large lateness due to its earlier deadline.

Therefore, swapping an adjacent inverted pair does **not increase** $L$, and strictly reduces the number of inversions.
This contradicts the choice of $S^*$ as an optimal schedule with the fewest inversions.

Hence $S^*$ has no inversions, so EDF is optimal.

In [56]:
jobs = [
    Job(process_time=3, ddl=4, name="A"),
    Job(process_time=1, ddl=9, name="B"),
    Job(process_time=2, ddl=6, name="C"),
    Job(process_time=4, ddl=7, name="D"),
]

orders = {
    "EDF (by deadline)": edf_order(jobs),
    "SPT (by processing time)": shortest_processing_time_order(jobs),
    "LDF (latest deadline first)": latest_deadline_first_order(jobs),
}

for name, order in orders.items():
    print("\n" + name)
    print_schedule_table(jobs, order)


EDF (by deadline)
Order: ['A', 'C', 'D', 'B']
job        t    d  start  finish   late
A          3    4      0       3      0
C          2    6      3       5      0
D          4    7      5       9      2
B          1    9      9      10      1
Max lateness L = 2

SPT (by processing time)
Order: ['B', 'C', 'A', 'D']
job        t    d  start  finish   late
B          1    9      0       1      0
C          2    6      1       3      0
A          3    4      3       6      2
D          4    7      6      10      3
Max lateness L = 3

LDF (latest deadline first)
Order: ['B', 'D', 'C', 'A']
job        t    d  start  finish   late
B          1    9      0       1      0
D          4    7      1       5      0
C          2    6      5       7      1
A          3    4      7      10      6
Max lateness L = 6


### 4.5 Random brute force verification (small n)

In [57]:
def brute_force_min_max_lateness(jobs: List[Job]) -> Tuple[int, List[int]]:
    best_L = None
    best_order = None
    for perm in itertools.permutations(range(len(jobs))):
        L, _ = evaluate_schedule(jobs, list(perm))
        if best_L is None or L < best_L:
            best_L = L
            best_order = list(perm)
    return best_L, best_order

def random_jobs(n: int, tmax=6, dmax=25) -> List[Job]:
    js = []
    for i in range(n):
        t = random.randint(1, tmax)
        d = random.randint(1, dmax)
        js.append(Job(process_time=t, ddl=d, name=f"J{i}"))
    return js

trials = 200
n = 0
bad = 0
for _ in range(trials):
    jobs = random_jobs(n)
    edf = edf_order(jobs)
    L_edf, _ = evaluate_schedule(jobs, edf)
    L_opt, _ = brute_force_min_max_lateness(jobs)
    if L_edf != L_opt:
        bad += 1

print(f"{bad} bad cases")

0 bad cases


## 5. Interval Partitioning (Minimum number of rooms)

### 5.1 Problem
We are given `n` lectures/intervals. Lecture $i$ has start time $s_i$ and finish time $f_i$ with $s_i < f_i$.

We must assign each lecture to a room such that no two overlapping lectures are in the same room.

Goal: **minimize** the number of rooms used.

---

### 5.2 Depth (lower bound)
Define the **depth** of a set of intervals as:
$$
\text{depth} = \max_{t} \left|\{ i : s_i \le t < f_i \}\right|.
$$
That is, the maximum number of intervals overlapping at any time.

**Lower bound:** Any valid assignment needs at least `depth` rooms, because at a time when `depth` intervals overlap, they must occupy distinct rooms.
So:
$$
\text{OPT} \ge \text{depth}.
$$

In [58]:
@dataclass(frozen=True)
class Lecture:
    start: int
    finish: int
    name: str = ""

    @property
    def duration(self) -> int:
        return self.finish - self.start

---
### 5.3 Greedy algorithm (start-time order + reuse a free room)

Algorithm:
1. Sort lectures by increasing start time.
2. When a lecture starts:
   - if there exists a room whose current lecture finishes at or before this start time, reuse that room;
   - otherwise open a new room.

Efficiently, maintain a **priority queue (min-heap)** keyed by each room's current finish time.

In [59]:
def interval_partitioning(lectures: List[Lecture]) -> Tuple[int, Dict[int, List[int]]]:
    """
    Returns:
      room_count: number of rooms used
      rooms: dict room_id -> list of lecture indices (in assigned order)

    Greedy:
      - sort by start time
      - reuse room with smallest finish time if available, else open new room
    """
    indexed = list(enumerate(lectures))
    indexed.sort(key=lambda x: x[1].start)

    # heap stores (finish_time, room_id)
    heap: List[Tuple[int, int]] = []
    rooms: Dict[int, List[int]] = {}
    next_room_id = 0

    for idx, lec in indexed:
        if heap and heap[0][0] <= lec.start:
            # reuse earliest-finishing room
            finish_time, room_id = heapq.heappop(heap)
        else:
            # open new room
            room_id = next_room_id
            rooms[room_id] = []
            next_room_id += 1

        rooms[room_id].append(idx)
        heapq.heappush(heap, (lec.finish, room_id))

    return next_room_id, rooms

def compute_depth(lectures: List[Lecture]) -> int:
    """
    Compute depth using a sweep line:
        +1 at start, -1 at finish (treat intervals as [start, finish])
    """
    events = []
    for lec in lectures:
        events.append((lec.start, +1))
        events.append((lec.finish, -1))

    # Sort by time; important: process finishes (-1) before starts (+1) at same time
    # because intervals are [start, finish): finishing at t frees before starting at t.
    events.sort(key=lambda x: (x[0], x[1]))

    cur = 0
    best = 0
    for _, delta in events:
        cur += delta
        best = max(best, cur)
    return best

---
### 5.4 Correctness proof (lower bound + greedy achieves it)

We already have the lower bound:
$$
\text{OPT} \ge \text{depth}.
$$

Now show the greedy algorithm uses exactly `depth` rooms.

Let the greedy algorithm open $R$ rooms total.
Consider the moment when greedy opens the $R$-th room (i.e., it cannot reuse any existing room).

At that moment, we are scheduling some lecture $i$ with start time $s_i$.
Greedy opens a new room because **every existing room** has a lecture whose finish time is $> s_i$.
Thus, at time $s_i$, there are already $R-1$ lectures in progress (one in each of the existing rooms),
and lecture $i$ also overlaps with them.

So at time $s_i$, there are at least $R$ overlapping lectures:
$$
\text{depth} \ge R.
$$

Combine with the lower bound $\text{OPT} \ge \text{depth}$:
$$
\text{OPT} \ge \text{depth} \ge R.
$$
But greedy uses $R$ rooms, so $R \ge \text{OPT}$ as well (since OPT is minimum).

Hence:
$$
R = \text{OPT}.
$$
So greedy is optimal.

In [60]:
lectures = [
    Lecture(start=0, finish=4, name="L0"),
    Lecture(start=1, finish=5, name="L1"),
    Lecture(start=2, finish=6, name="L2"),
    Lecture(start=4, finish=7, name="L3"),
    Lecture(start=5, finish=9, name="L4"),
    Lecture(start=6, finish=10, name="L5"),
]

room_count, rooms = interval_partitioning(lectures)
depth = compute_depth(lectures)

print("Depth =", depth)
print("Greedy rooms used =", room_count)

for r in sorted(rooms):
    print(f"Room {r}:", [lectures[i].name for i in rooms[r]])

Depth = 3
Greedy rooms used = 3
Room 0: ['L0', 'L3']
Room 1: ['L1', 'L4']
Room 2: ['L2', 'L5']


---
### 5.5 Random stress test: greedy rooms == depth

In [61]:
def random_lectures(n: int, tmax=30) -> List[Lecture]:
    lecs = []
    for i in range(n):
        s = random.randint(0, tmax-1)
        f = random.randint(s+1, tmax)
        lecs.append(Lecture(start=s, finish=f, name=f"R{i}"))
    return lecs

trials = 300
n = 20
bad = 0

for _ in range(trials):
    lecs = random_lectures(n)
    R, _ = interval_partitioning(lecs)
    dep = compute_depth(lecs)
    if R != dep:
        bad += 1

print(f"{bad} bad cases")

0 bad cases
